# FinBERT Aspect Classifier — Colab Runner

This notebook clones the repo, installs dependencies, uploads your data, and runs training.  
All heavy logic lives in the `.py` files — this notebook is just the launcher.

**Before running:** Runtime → Change runtime type → **T4 GPU**


## 1 · Clone repo & install

In [ ]:
import os

REPO_URL    = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"  # <- change this
BRANCH      = "main"
REPO_DIR    = "/content/finbert"

# Clone (or pull if already cloned)
if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull origin {BRANCH}

%cd {REPO_DIR}
!pip install -r requirements.txt --quiet


## 2 · Mount Google Drive
*Checkpoints and outputs survive disconnection here.*

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

OUTPUT_DIR = '/content/drive/MyDrive/finbert_outputs'
import os; os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Outputs → {OUTPUT_DIR}")


## 3 · Upload your data

In [ ]:
from google.colab import files

uploaded  = files.upload()           # select your labels.csv
DATA_PATH = list(uploaded.keys())[0]
print(f"Uploaded: {DATA_PATH}")

# Quick sanity check
import pandas as pd
df = pd.read_csv(DATA_PATH)
print(f"{len(df)} rows  |  columns: {list(df.columns)}")
df.head(3)


## 4 · Verify GPU

In [ ]:
!nvidia-smi | head -15
import torch
print(f"\nPyTorch sees CUDA: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")


## 5 · Edit config (optional)

Open `config.py` in the repo and change hyperparameters there.  
The defaults are already set correctly based on your data diagnostics:
- Fiscal `pos_weight` capped at 8
- Abstain zone masked at [0.35, 0.55]  
- 3-phase training with early stopping

Re-run cell 1 (`git pull`) after any edits pushed to GitHub.


## 6 · Train

In [ ]:
# Pass OUTPUT_DIR so checkpoints go to Drive
%cd {REPO_DIR}

!python train.py --data "{DATA_PATH}" --output_dir "{OUTPUT_DIR}"


## 7 · Plot results

In [ ]:
%cd {REPO_DIR}
!python utils/plot.py --output_dir "{OUTPUT_DIR}"

# Display inline
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

for fname in ['training_curves.png', 'comparison_chart.png']:
    path = f"{OUTPUT_DIR}/{fname}"
    import os
    if os.path.exists(path):
        img = mpimg.imread(path)
        plt.figure(figsize=(14, 6))
        plt.imshow(img); plt.axis('off')
        plt.title(fname); plt.show()


## 8 · Download results

In [ ]:
from google.colab import files

for fname in ['comparison_results.csv', 'comparison_chart.png', 'training_curves.png']:
    path = f"{OUTPUT_DIR}/{fname}"
    if os.path.exists(path):
        files.download(path)
        print(f"Downloaded: {fname}")
